# VWAP Cross Model — Hypothesis Validation

**Thesis:** Weekly VWAP and Daily VWAP confluence zones act as institutional structural levels. When price crosses both VWAPs in alignment, with RSI confirming the direction, there is a tradeable momentum edge.

**Questions to answer:**
1. Does weekly VWAP deviation predict forward returns?
2. Does weekly/daily VWAP cross generate alpha over buy-and-hold?
3. Does RSI filtering improve signal quality?
4. What is the optimal VWAP period (rolling 7d vs fixed Monday)?
5. Is the signal orthogonal to existing MeanReversion/Momentum models?

**Data:** Binance Futures OHLCV (no DB needed, direct API).

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

from binance.um_futures import UMFutures
from libs.features.indicators.momentum.rsi import RSI, _compute_rsi_batch
from libs.features.indicators.volatility.atr import ATR, _compute_atr_batch

plt.style.use('dark_background')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup complete')

## 1. Data Fetch

In [ ]:
# ── Binance Futures paginated fetch ──────────────────────────────────
_RAW_COLS = [
    "timestamp", "open", "high", "low", "close", "volume", "close_time",
    "quote_vol", "trades", "taker_buy_base", "taker_buy_quote", "ignore",
]
OHLCV_COLS = ["timestamp", "open", "high", "low", "close", "volume"]
_MAX_LIMIT = 1500

def fetch_ohlcv(symbol: str, timeframe: str, days: int = 365) -> pd.DataFrame:
    """Fetch historical OHLCV from Binance Futures with auto-pagination."""
    client = UMFutures()
    now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
    since = now_ms - days * 86_400_000
    end = now_ms
    frames = []
    cursor = since
    while cursor < end:
        lines = client.klines(symbol, timeframe, startTime=cursor, endTime=end, limit=_MAX_LIMIT)
        if not lines:
            break
        df = pd.DataFrame(lines, columns=_RAW_COLS)[OHLCV_COLS]
        for c in OHLCV_COLS:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        frames.append(df)
        last_ts = int(df['timestamp'].iloc[-1])
        if last_ts <= cursor:
            break
        cursor = last_ts + 1
        if len(lines) < _MAX_LIMIT:
            break
    if not frames:
        return pd.DataFrame(columns=OHLCV_COLS)
    result = pd.concat(frames, ignore_index=True)
    result = result.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)
    result['dt'] = pd.to_datetime(result['timestamp'], unit='ms', utc=True)
    result = result.set_index('dt')
    print(f"{symbol} {timeframe}: {len(result)} candles ({result.index[0].date()} to {result.index[-1].date()})")
    return result

# Fetch 1 year of data for statistical significance
btc_1h = fetch_ohlcv('BTCUSDT', '1h', days=365)
eth_1h = fetch_ohlcv('ETHUSDT', '1h', days=365)
btc_15m = fetch_ohlcv('BTCUSDT', '15m', days=180)

## 2. VWAP Construction

Build Daily VWAP and Weekly VWAP (rolling 7-day and fixed Monday-anchor variants).

In [ ]:
def compute_vwap_rolling(df: pd.DataFrame, period_hours: int) -> pd.Series:
    """Rolling VWAP over a fixed number of bars."""
    tp = (df['high'] + df['low'] + df['close']) / 3.0
    tp_vol = tp * df['volume']
    cum_tp_vol = tp_vol.rolling(period_hours, min_periods=1).sum()
    cum_vol = df['volume'].rolling(period_hours, min_periods=1).sum()
    return cum_tp_vol / cum_vol.replace(0, np.nan)


def compute_vwap_anchored(df: pd.DataFrame, anchor: str = 'D') -> pd.Series:
    """Session-anchored VWAP. anchor='D' for daily, 'W-MON' for weekly Monday."""
    tp = (df['high'] + df['low'] + df['close']) / 3.0
    tp_vol = tp * df['volume']
    # Group by anchor period
    groups = df.index.to_period(anchor[0])  # 'D' or 'W'
    cum_tp_vol = tp_vol.groupby(groups).cumsum()
    cum_vol = df['volume'].groupby(groups).cumsum()
    return cum_tp_vol / cum_vol.replace(0, np.nan)


def add_vwap_features(df: pd.DataFrame, tf_hours: float = 1.0) -> pd.DataFrame:
    """Add all VWAP variants as columns."""
    out = df.copy()
    bars_per_day = int(24 / tf_hours)
    bars_per_week = bars_per_day * 7
    
    # Rolling VWAPs
    out['vwap_daily_rolling'] = compute_vwap_rolling(out, bars_per_day)
    out['vwap_weekly_rolling'] = compute_vwap_rolling(out, bars_per_week)
    
    # Anchored VWAPs
    out['vwap_daily_anchored'] = compute_vwap_anchored(out, 'D')
    out['vwap_weekly_anchored'] = compute_vwap_anchored(out, 'W')
    
    # Deviations (normalized by price)
    out['vwap_daily_dev'] = (out['close'] - out['vwap_daily_rolling']) / out['close'] * 100
    out['vwap_weekly_dev'] = (out['close'] - out['vwap_weekly_rolling']) / out['close'] * 100
    
    # Cross signals: price crosses weekly VWAP
    out['above_weekly_vwap'] = (out['close'] > out['vwap_weekly_rolling']).astype(int)
    out['weekly_vwap_cross_up'] = out['above_weekly_vwap'].diff().clip(lower=0)  # 0->1
    out['weekly_vwap_cross_dn'] = (-out['above_weekly_vwap'].diff()).clip(lower=0)  # 1->0
    
    # VWAP alignment: both daily and weekly on same side
    out['above_daily_vwap'] = (out['close'] > out['vwap_daily_rolling']).astype(int)
    out['vwap_aligned_bull'] = ((out['above_weekly_vwap'] == 1) & (out['above_daily_vwap'] == 1)).astype(int)
    out['vwap_aligned_bear'] = ((out['above_weekly_vwap'] == 0) & (out['above_daily_vwap'] == 0)).astype(int)
    
    return out

btc_1h = add_vwap_features(btc_1h, tf_hours=1.0)
eth_1h = add_vwap_features(eth_1h, tf_hours=1.0)
print(f"VWAP features computed. Columns: {[c for c in btc_1h.columns if 'vwap' in c]}")

## 3. Forward Returns & Signal Quality

In [ ]:
def add_forward_returns(df: pd.DataFrame, horizons: list[int] = [1, 4, 12, 24]) -> pd.DataFrame:
    """Add forward returns at multiple horizons (in bars)."""
    out = df.copy()
    for h in horizons:
        out[f'fwd_ret_{h}'] = out['close'].pct_change(h).shift(-h)
    return out

btc_1h = add_forward_returns(btc_1h)
eth_1h = add_forward_returns(eth_1h)

# ── Test 1: VWAP deviation vs forward returns (predictive power) ────
print("=" * 60)
print("VWAP DEVIATION vs FORWARD RETURNS (Pearson correlation)")
print("=" * 60)
for label, data in [('BTC/1h', btc_1h), ('ETH/1h', eth_1h)]:
    print(f"\n{label}:")
    for dev_col in ['vwap_daily_dev', 'vwap_weekly_dev']:
        for ret_col in ['fwd_ret_1', 'fwd_ret_4', 'fwd_ret_12', 'fwd_ret_24']:
            corr = data[[dev_col, ret_col]].dropna().corr().iloc[0, 1]
            print(f"  {dev_col:20s} vs {ret_col:12s}: r = {corr:+.4f}")

In [ ]:
# ── Test 2: Weekly VWAP Cross signal — conditional forward returns ──
print("=" * 60)
print("WEEKLY VWAP CROSS — Conditional Forward Returns")
print("=" * 60)

for label, data in [('BTC/1h', btc_1h), ('ETH/1h', eth_1h)]:
    print(f"\n{label}:")
    cross_up = data[data['weekly_vwap_cross_up'] == 1]
    cross_dn = data[data['weekly_vwap_cross_dn'] == 1]
    
    print(f"  Cross UP events: {len(cross_up)}")
    print(f"  Cross DN events: {len(cross_dn)}")
    
    for h in [4, 12, 24]:
        ret_col = f'fwd_ret_{h}'
        up_ret = cross_up[ret_col].dropna()
        dn_ret = cross_dn[ret_col].dropna()
        
        up_mean = up_ret.mean() * 100
        up_wr = (up_ret > 0).mean() * 100
        dn_mean = dn_ret.mean() * 100
        dn_wr = (dn_ret < 0).mean() * 100
        
        print(f"  {h}h: UP→ mean={up_mean:+.3f}% WR={up_wr:.1f}% | DN→ mean={dn_mean:+.3f}% WR={dn_wr:.1f}%")

In [ ]:
# ── Test 3: VWAP Alignment + RSI filter ──────────────────────────────
rsi_vals = _compute_rsi_batch(btc_1h['close'].values, 14)
btc_1h['RSI'] = rsi_vals

rsi_vals_eth = _compute_rsi_batch(eth_1h['close'].values, 14)
eth_1h['RSI'] = rsi_vals_eth

print("=" * 60)
print("VWAP ALIGNMENT + RSI FILTER — Conditional Forward Returns")
print("=" * 60)

for label, data in [('BTC/1h', btc_1h), ('ETH/1h', eth_1h)]:
    print(f"\n{label}:")
    
    # Long: both VWAPs aligned bullish + RSI < 70 (not overbought)
    long_signal = data[(data['vwap_aligned_bull'] == 1) & (data['RSI'] < 70) & (data['RSI'] > 40)]
    # Short: both VWAPs aligned bearish + RSI > 30 (not oversold)  
    short_signal = data[(data['vwap_aligned_bear'] == 1) & (data['RSI'] > 30) & (data['RSI'] < 60)]
    
    print(f"  Long signals (aligned bull + RSI 40-70): {len(long_signal)}")
    print(f"  Short signals (aligned bear + RSI 30-60): {len(short_signal)}")
    
    for h in [4, 12, 24]:
        ret_col = f'fwd_ret_{h}'
        l_ret = long_signal[ret_col].dropna()
        s_ret = short_signal[ret_col].dropna()
        
        if len(l_ret) > 0:
            print(f"  {h}h LONG:  mean={l_ret.mean()*100:+.3f}% WR={((l_ret>0).mean())*100:.1f}% n={len(l_ret)}")
        if len(s_ret) > 0:
            print(f"  {h}h SHORT: mean={s_ret.mean()*100:+.3f}% WR={((s_ret<0).mean())*100:.1f}% n={len(s_ret)}")

## 4. Simple Backtest — VWAP Cross with Multi-TP

In [ ]:
# ── Simple vectorized backtest ─────────────────────────────────────

def backtest_vwap_cross(
    df: pd.DataFrame,
    rsi_long_max: float = 70,
    rsi_short_min: float = 30,
    atr_tp_mult: float = 2.0,
    atr_sl_mult: float = 1.5,
    cooldown_bars: int = 12,
) -> pd.DataFrame:
    """Bar-by-bar backtest of VWAP cross signals with ATR-based TP/SL."""
    # Compute ATR
    hlc = np.column_stack([df['high'].values, df['low'].values, df['close'].values])
    atr_vals = _compute_atr_batch(hlc[:, 0], hlc[:, 1], hlc[:, 2], 14)
    
    closes = df['close'].values
    highs = df['high'].values
    lows = df['low'].values
    rsi = df['RSI'].values
    cross_up = df['weekly_vwap_cross_up'].values
    cross_dn = df['weekly_vwap_cross_dn'].values
    aligned_bull = df['vwap_aligned_bull'].values
    aligned_bear = df['vwap_aligned_bear'].values
    
    trades = []
    position = 0  # 0=flat, 1=long, -1=short
    entry_price = 0.0
    tp_price = 0.0
    sl_price = 0.0
    entry_bar = 0
    last_exit_bar = -cooldown_bars
    
    for i in range(50, len(df)):
        if np.isnan(atr_vals[i]) or np.isnan(rsi[i]):
            continue
            
        atr = atr_vals[i]
        
        # Check exits first
        if position == 1:
            if lows[i] <= sl_price:
                pnl = (sl_price - entry_price) / entry_price * 100
                trades.append({'entry_bar': entry_bar, 'exit_bar': i, 'direction': 'long', 'pnl_pct': pnl, 'reason': 'SL'})
                position = 0
                last_exit_bar = i
            elif highs[i] >= tp_price:
                pnl = (tp_price - entry_price) / entry_price * 100
                trades.append({'entry_bar': entry_bar, 'exit_bar': i, 'direction': 'long', 'pnl_pct': pnl, 'reason': 'TP'})
                position = 0
                last_exit_bar = i
        elif position == -1:
            if highs[i] >= sl_price:
                pnl = (entry_price - sl_price) / entry_price * 100
                trades.append({'entry_bar': entry_bar, 'exit_bar': i, 'direction': 'short', 'pnl_pct': pnl, 'reason': 'SL'})
                position = 0
                last_exit_bar = i
            elif lows[i] <= tp_price:
                pnl = (entry_price - tp_price) / entry_price * 100
                trades.append({'entry_bar': entry_bar, 'exit_bar': i, 'direction': 'short', 'pnl_pct': pnl, 'reason': 'TP'})
                position = 0
                last_exit_bar = i
        
        # Check entries (only if flat and past cooldown)
        if position == 0 and (i - last_exit_bar) >= cooldown_bars:
            # Long: weekly VWAP cross up OR aligned bull, RSI confirmation
            if (cross_up[i] == 1 or (aligned_bull[i] == 1 and aligned_bull[i-1] == 0)) and rsi[i] < rsi_long_max and rsi[i] > 30:
                position = 1
                entry_price = closes[i]
                tp_price = entry_price + atr * atr_tp_mult
                sl_price = entry_price - atr * atr_sl_mult
                entry_bar = i
            # Short: weekly VWAP cross down OR aligned bear, RSI confirmation
            elif (cross_dn[i] == 1 or (aligned_bear[i] == 1 and aligned_bear[i-1] == 0)) and rsi[i] > rsi_short_min and rsi[i] < 70:
                position = -1
                entry_price = closes[i]
                tp_price = entry_price - atr * atr_tp_mult
                sl_price = entry_price + atr * atr_sl_mult
                entry_bar = i
    
    return pd.DataFrame(trades)

# Run backtest
trades_btc = backtest_vwap_cross(btc_1h)
trades_eth = backtest_vwap_cross(eth_1h)

print("=" * 60)
print("VWAP CROSS BACKTEST RESULTS")
print("=" * 60)
for label, trades in [('BTC/1h', trades_btc), ('ETH/1h', trades_eth)]:
    if trades.empty:
        print(f"\n{label}: No trades")
        continue
    n = len(trades)
    win_rate = (trades['pnl_pct'] > 0).mean() * 100
    avg_pnl = trades['pnl_pct'].mean()
    total_pnl = trades['pnl_pct'].sum()
    avg_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].mean() if (trades['pnl_pct'] > 0).any() else 0
    avg_loss = trades[trades['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades['pnl_pct'] <= 0).any() else 0
    
    print(f"\n{label}:")
    print(f"  Trades: {n}  |  Win Rate: {win_rate:.1f}%")
    print(f"  Avg PnL: {avg_pnl:+.3f}%  |  Total PnL: {total_pnl:+.2f}%")
    print(f"  Avg Win: {avg_win:+.3f}%  |  Avg Loss: {avg_loss:+.3f}%")
    print(f"  Profit Factor: {abs(avg_win/avg_loss) if avg_loss != 0 else 'inf':.2f}")
    print(f"  By reason: {trades['reason'].value_counts().to_dict()}")

In [ ]:
# ── Equity curve visualization ────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ax, (label, trades) in zip(axes, [('BTC/1h', trades_btc), ('ETH/1h', trades_eth)]):
    if trades.empty:
        ax.set_title(f"{label}: No trades")
        continue
    cum_pnl = trades['pnl_pct'].cumsum()
    ax.plot(cum_pnl.values, linewidth=1.5)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f"{label} — VWAP Cross Cumulative PnL ({len(trades)} trades)")
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative PnL (%)')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Parameter Sensitivity & Robustness

In [ ]:
# ── ATR multiplier sweep ──────────────────────────────────────────
results = []
for tp_mult in [1.5, 2.0, 2.5, 3.0, 4.0]:
    for sl_mult in [1.0, 1.5, 2.0, 2.5]:
        trades = backtest_vwap_cross(btc_1h, atr_tp_mult=tp_mult, atr_sl_mult=sl_mult)
        if trades.empty:
            continue
        results.append({
            'tp_mult': tp_mult,
            'sl_mult': sl_mult,
            'n_trades': len(trades),
            'win_rate': (trades['pnl_pct'] > 0).mean() * 100,
            'total_pnl': trades['pnl_pct'].sum(),
            'avg_pnl': trades['pnl_pct'].mean(),
            'sharpe': trades['pnl_pct'].mean() / trades['pnl_pct'].std() * np.sqrt(len(trades)) if trades['pnl_pct'].std() > 0 else 0,
        })

sweep_df = pd.DataFrame(results).sort_values('sharpe', ascending=False)
print("ATR TP/SL Multiplier Sweep (BTC/1h):")
print(sweep_df.head(10).to_string(index=False))

## 6. Conclusion

**Decision criteria:**
- If correlation between VWAP deviation and forward returns is statistically significant (|r| > 0.03 with p < 0.05)
- AND weekly VWAP cross produces win rate > 55% with positive expectancy
- AND results hold on both BTC and ETH (out-of-sample asset)

→ **Proceed to full model build**

Otherwise → **Abandon or pivot the thesis**

In [ ]:
# ── Summary of all key results ────────────────────────────────────
print("=" * 70)
print("VWAP CROSS HYPOTHESIS — FULL RESULTS SUMMARY")
print("=" * 70)

# 1. Correlation
print("\n▸ VWAP Deviation vs Forward Returns (Pearson r):")
for label, data in [('BTC/1h', btc_1h), ('ETH/1h', eth_1h)]:
    print(f"  {label}:")
    for dev_col in ['vwap_daily_dev', 'vwap_weekly_dev']:
        for ret_col in ['fwd_ret_4', 'fwd_ret_12', 'fwd_ret_24']:
            corr = data[[dev_col, ret_col]].dropna().corr().iloc[0, 1]
            print(f"    {dev_col:20s} vs {ret_col}: r={corr:+.4f}")

# 2. Cross signals
print("\n▸ Weekly VWAP Cross — Conditional Fwd Returns:")
for label, data in [('BTC/1h', btc_1h), ('ETH/1h', eth_1h)]:
    cross_up = data[data['weekly_vwap_cross_up'] == 1]
    cross_dn = data[data['weekly_vwap_cross_dn'] == 1]
    print(f"  {label}: UP={len(cross_up)} DN={len(cross_dn)}")
    for h in [4, 12, 24]:
        up_ret = cross_up[f'fwd_ret_{h}'].dropna()
        dn_ret = cross_dn[f'fwd_ret_{h}'].dropna()
        print(f"    {h}h: UP mean={up_ret.mean()*100:+.3f}% WR={((up_ret>0).mean())*100:.1f}% | DN mean={dn_ret.mean()*100:+.3f}% WR={((dn_ret<0).mean())*100:.1f}%")

# 3. Alignment + RSI
print("\n▸ VWAP Alignment + RSI Filter:")
for label, data in [('BTC/1h', btc_1h), ('ETH/1h', eth_1h)]:
    long_sig = data[(data['vwap_aligned_bull'] == 1) & (data['RSI'] < 70) & (data['RSI'] > 40)]
    short_sig = data[(data['vwap_aligned_bear'] == 1) & (data['RSI'] > 30) & (data['RSI'] < 60)]
    print(f"  {label}: Long={len(long_sig)} Short={len(short_sig)}")
    for h in [12, 24]:
        l_r = long_sig[f'fwd_ret_{h}'].dropna()
        s_r = short_sig[f'fwd_ret_{h}'].dropna()
        if len(l_r) > 0:
            print(f"    {h}h LONG:  mean={l_r.mean()*100:+.3f}% WR={((l_r>0).mean())*100:.1f}% n={len(l_r)}")
        if len(s_r) > 0:
            print(f"    {h}h SHORT: mean={s_r.mean()*100:+.3f}% WR={((s_r<0).mean())*100:.1f}% n={len(s_r)}")

# 4. Backtest
print("\n▸ Backtest Results:")
for label, trades in [('BTC/1h', trades_btc), ('ETH/1h', trades_eth)]:
    if trades.empty:
        print(f"  {label}: No trades"); continue
    n = len(trades)
    wr = (trades['pnl_pct'] > 0).mean() * 100
    avg = trades['pnl_pct'].mean()
    tot = trades['pnl_pct'].sum()
    aw = trades[trades['pnl_pct'] > 0]['pnl_pct'].mean() if (trades['pnl_pct'] > 0).any() else 0
    al = trades[trades['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades['pnl_pct'] <= 0).any() else 0
    pf = abs(aw/al) if al != 0 else float('inf')
    print(f"  {label}: {n} trades | WR={wr:.1f}% | Avg={avg:+.3f}% | Total={tot:+.2f}% | PF={pf:.2f}")
    print(f"    TP/SL: {trades['reason'].value_counts().to_dict()}")

# 5. Sweep best
print("\n▸ Top 5 ATR Sweep (BTC/1h):")
print(sweep_df.head(5).to_string(index=False))